# **Single-Cell RNA-Seq Analysis Project**

In this project, you will work with the `norman` dataset from the `perturbation_data_analysis` exercise:

In [1]:
import os
import sys
from pathlib import Path
import scanpy as sc

project_root = Path().absolute().parent
# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

import pertdata as pt  # noqa: E402

# Use the existing norman dataset location
norman = pt.PertDataset(name="norman", cache_dir_path="../data", silent=False)

print(norman)

Dataset already cached: d:\IML\Genomic-Data-Science\data\norman
Loading: d:\IML\Genomic-Data-Science\data\norman\norman\perturb_processed.h5ad
PertDataset object
    name: norman
    cache_dir_path: d:\IML\Genomic-Data-Science\data
    path: d:\IML\Genomic-Data-Science\data\norman
    adata: AnnData object with n_obs ✕ n_vars = 91205 ✕ 5045


In [2]:

adata = sc.read_h5ad('../data/norman/norman/perturb_processed.h5ad')
print(f"Dataset loaded: {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"Available metadata: {list(adata.obs.columns)}")

Dataset loaded: 91205 cells × 5045 genes
Available metadata: ['condition', 'cell_type', 'dose_val', 'control', 'condition_name']


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from src.models.svm_model import SVMModel
import numpy as np

# Keep data SPARSE - don't convert to dense array!
X = adata.X  # Keep as sparse matrix
print(f"Gene expression matrix: {X.shape} (sparse: {type(X).__name__})")

y_labels = adata.obs['condition'].values
print(f"Labels shape: {y_labels.shape}")
print(f"Unique perturbations: {len(set(y_labels))}")
print(f"Sample labels: {y_labels[:5]}")

# Encode string labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)
print(f"\nEncoded labels range: {y.min()} to {y.max()}")

# Split into train/test (80/20) - NO SUBSAMPLING, using full dataset
print("\n" + "="*50)
print("Using FULL dataset (sparse format)")
print("="*50)
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_seed, stratify=y
)

print(f"\nTraining data: X_train.shape={X_train.shape}")
print(f"Test data: X_test.shape={X_test.shape}")
print(f"Memory usage: ~{X_train.data.nbytes / 1e6:.1f} MB (sparse)")

# Create and train SVM with sparse support
print("\n" + "="*50)
print("Training LinearSVM (memory efficient)...")
print("="*50)
model = SVMModel(
    kernel='linear',      # LinearSVC is much faster & memory efficient
    C=1.0, 
    pca_components=100,   # Dimensionality reduction
    use_sparse=True,      # Don't center sparse matrices
    random_state=random_seed
)
model.fit(X_train, y_train)

# Evaluate on test set
print("\nEvaluating on test set...")
results = model.evaluate(X_test, y_test)
print("\nTest Results:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

Gene expression matrix: (91205, 5045) (sparse: csr_matrix)
Labels shape: (91205,)
Unique perturbations: 284
Sample labels: ['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl']
Categories (284, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

Encoded labels range: 0 to 283

Using FULL dataset (sparse format)

Training data: X_train.shape=(72964, 5045)
Test data: X_test.shape=(18241, 5045)
Memory usage: ~119.4 MB (sparse)

Training LinearSVM (memory efficient)...


d:\IML\Genomic-Data-Science\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Evaluating on test set...

Test Results:
  accuracy: 0.2491
  precision: 0.2008
  recall: 0.2491
  f1_score: 0.1753


d:\IML\Genomic-Data-Science\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [4]:
# Diagnose the problem
import pandas as pd

# Check class distribution
print("Class distribution:")
class_counts = pd.Series(y).value_counts()
print(f"Number of classes: {len(class_counts)}")
print(f"Samples per class (min/mean/max): {class_counts.min()}/{class_counts.mean():.1f}/{class_counts.max()}")
print(f"\nClasses with < 10 samples: {(class_counts < 10).sum()}")
print(f"Classes with < 50 samples: {(class_counts < 50).sum()}")

# Show some class distributions
print("\nTop 10 most common perturbations:")
label_names = label_encoder.inverse_transform(class_counts.index[:10])
for name, count in zip(label_names, class_counts.values[:10]):
    print(f"  {name}: {count} samples")

print("\n" + "="*70)
print("PROBLEM: Too many classes with too few samples each!")
print("="*70)
print("\nSuggested fixes:")
print("1. Filter to only common perturbations (e.g., > 100 samples)")
print("2. Remove PCA or use more components (try 500+)")
print("3. Use different kernel or more regularization")

Class distribution:
Number of classes: 284
Samples per class (min/mean/max): 49/321.1/7353

Classes with < 10 samples: 0
Classes with < 50 samples: 1

Top 10 most common perturbations:
  ctrl: 7353 samples
  CEBPE+RUNX1T1: 1030 samples
  KLF1+ctrl: 997 samples
  TBX3+TBX2: 969 samples
  SLC4A1+ctrl: 853 samples
  ETS2+CNN1: 785 samples
  DUSP9+ETS2: 698 samples
  UBASH3B+OSR2: 677 samples
  DUSP9+ctrl: 662 samples
  ctrl+ETS2: 656 samples

PROBLEM: Too many classes with too few samples each!

Suggested fixes:
1. Filter to only common perturbations (e.g., > 100 samples)
2. Remove PCA or use more components (try 500+)
3. Use different kernel or more regularization


In [10]:
# UNDERSTAND THE FILTERING TRADE-OFF
import pandas as pd
import matplotlib.pyplot as plt

print("="*70)
print("FILTERING ANALYSIS: What are we keeping vs discarding?")
print("="*70)

# Analyze the full dataset
all_counts = pd.Series(y_labels).value_counts()
print(f"\n📊 FULL DATASET:")
print(f"  Total perturbations: {len(all_counts)}")
print(f"  Total samples: {len(y_labels):,}")

# Different filtering thresholds
thresholds = [100, 200, 500, 1000]

print("\n" + "="*70)
print("IMPACT OF DIFFERENT FILTERING THRESHOLDS:")
print("="*70)

for threshold in thresholds:
    kept_classes = (all_counts >= threshold).sum()
    discarded_classes = (all_counts < threshold).sum()
    kept_samples = all_counts[all_counts >= threshold].sum()
    discarded_samples = all_counts[all_counts < threshold].sum()
    
    print(f"\nmin_samples = {threshold}:")
    print(f"  ✓ Kept:      {kept_classes:3d} classes ({kept_samples:,} samples = {kept_samples/len(y_labels)*100:.1f}%)")
    print(f"  ✗ Discarded: {discarded_classes:3d} classes ({discarded_samples:,} samples = {discarded_samples/len(y_labels)*100:.1f}%)")

# Show what specific perturbations we're losing with min_samples=500
print("\n" + "="*70)
print("PERTURBATIONS DISCARDED (min_samples=500):")
print("="*70)

kept_mask = all_counts >= 500
discarded_perturbations = all_counts[~kept_mask].sort_values(ascending=False)

print(f"\nTop 20 discarded perturbations (have 100-499 samples):")
for pert, count in discarded_perturbations[discarded_perturbations >= 100].head(20).items():
    print(f"  {pert}: {count} samples")

if len(discarded_perturbations[discarded_perturbations < 100]) > 0:
    print(f"\n...and {len(discarded_perturbations[discarded_perturbations < 100])} more with <100 samples each")

# Kept perturbations
print("\n" + "="*70)
print("PERTURBATIONS KEPT (min_samples=500):")
print("="*70)
kept_perturbations = all_counts[kept_mask].sort_values(ascending=False)
print(f"\nAll {len(kept_perturbations)} kept perturbations:")
for pert, count in kept_perturbations.items():
    print(f"  {pert}: {count} samples")

print("\n" + "="*70)
print("TRADE-OFF SUMMARY:")
print("="*70)
print("✓ PROS of filtering (min_samples=500):")
print("  • Much better accuracy (69.7% vs 2.5%)")
print("  • Model can actually learn patterns")
print("  • Faster training")
print("  • More balanced classes")
print("\n✗ CONS of filtering:")
print("  • Can't classify rare perturbations")
print("  • Lose data from uncommon experiments")
print("  • Bias toward well-studied perturbations")

print("\n💡 ALTERNATIVES:")
print("  1. Keep all classes, use class_weight='balanced' in SVM")
print("  2. Use hierarchical classification (group rare perturbations)")
print("  3. Two-stage model: common vs rare, then classify within each")
print("  4. Different threshold (e.g., min_samples=200)")
print("  5. Data augmentation for rare classes")

FILTERING ANALYSIS: What are we keeping vs discarding?

📊 FULL DATASET:
  Total perturbations: 284
  Total samples: 91,205

IMPACT OF DIFFERENT FILTERING THRESHOLDS:

min_samples = 100:
  ✓ Kept:      267 classes (89,917 samples = 98.6%)
  ✗ Discarded:  17 classes (1,288 samples = 1.4%)

min_samples = 200:
  ✓ Kept:      214 classes (81,412 samples = 89.3%)
  ✗ Discarded:  70 classes (9,793 samples = 10.7%)

min_samples = 500:
  ✓ Kept:       25 classes (22,882 samples = 25.1%)
  ✗ Discarded: 259 classes (68,323 samples = 74.9%)

min_samples = 1000:
  ✓ Kept:        2 classes (8,383 samples = 9.2%)
  ✗ Discarded: 282 classes (82,822 samples = 90.8%)

PERTURBATIONS DISCARDED (min_samples=500):

Top 20 discarded perturbations (have 100-499 samples):
  MAP2K3+ELMSAN1: 484 samples
  C19orf26+ctrl: 480 samples
  AHR+ctrl: 479 samples
  ctrl+FEV: 474 samples
  CEBPE+ctrl: 473 samples
  UBASH3B+ctrl: 470 samples
  BCL2L11+ctrl: 463 samples
  TMSB4X+ctrl: 462 samples
  MAP2K3+ctrl: 458 samples

In [13]:
# BETTER APPROACH: Handle Imbalance Without Discarding Data
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import TruncatedSVD
import time

print("="*70)
print("ALTERNATIVE: Keep ALL Classes with Class Weighting")
print("="*70)

print("\n🎯 GOAL: Classify ALL perturbations, not just common ones")
print("   Strategy: Use class_weight='balanced' to handle imbalance")

# Use ALL data - no filtering!
print("\n📊 Using FULL dataset:")
print(f"  Total perturbations: {len(np.unique(y_labels))}")
print(f"  Total samples: {len(y_labels):,}")

# Dimensionality reduction (necessary for tractability)
n_components = 200
print(f"\nReducing {X.shape[1]} genes -> {n_components} components...")
svd_full = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced_full = svd_full.fit_transform(X)
print(f"Explained variance: {svd_full.explained_variance_ratio_.sum():.2%}")

# Split data (using original y_labels, not filtered!)
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_reduced_full, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining data: {X_train_full.shape}")
print(f"Test data: {X_test_full.shape}")

# Train with balanced class weights
print("\n" + "="*50)
print("Training SVM with class_weight='balanced'...")
print("="*50)
print("This gives rare classes higher importance during training")

start_time = time.time()

full_model = LinearSVC(
    C=0.1,                      # Strong regularization for many classes
    loss='squared_hinge',
    dual=False,
    max_iter=2000,
    tol=1e-4,
    class_weight='balanced',    # KEY: Handle imbalance automatically!
    random_state=42,
    verbose=1
)

full_model.fit(X_train_full, y_train_full)

training_time = time.time() - start_time
print(f"\n✓ Training completed in {training_time:.1f} seconds")

# Evaluate
y_pred_full = full_model.predict(X_test_full)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

acc_full = accuracy_score(y_test_full, y_pred_full)
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(
    y_test_full, y_pred_full, average='weighted', zero_division=0
)

print("\n" + "="*50)
print("RESULTS: All Classes (No Filtering)")
print("="*50)
print(f"  accuracy:  {acc_full:.4f}")
print(f"  precision: {prec_full:.4f}")
print(f"  recall:    {rec_full:.4f}")
print(f"  f1_score:  {f1_full:.4f}")

# Comparison
print("\n" + "="*50)
print("COMPARISON: Filtered vs All Classes")
print("="*50)
if 'acc_hp' in locals():
    print(f"Filtered (25 classes):      {acc_hp:.4f}")
print(f"All classes ({len(np.unique(y_labels))} total): {acc_full:.4f}")

# Show per-class performance for rare vs common
print("\n" + "="*50)
print("Performance by Class Frequency:")
print("="*50)

# Get class frequencies
class_counts_series = pd.Series(y_train_full).value_counts()

# Get per-class metrics
report = classification_report(y_test_full, y_pred_full, output_dict=True, zero_division=0)

# Group by frequency
rare_classes = class_counts_series[class_counts_series < 100].index
common_classes = class_counts_series[class_counts_series >= 500].index
medium_classes = class_counts_series[(class_counts_series >= 100) & (class_counts_series < 500)].index

def avg_f1_for_classes(classes, report):
    f1_scores = [report[str(c)]['f1-score'] for c in classes if str(c) in report]
    return np.mean(f1_scores) if f1_scores else 0.0

rare_f1 = avg_f1_for_classes(rare_classes, report)
medium_f1 = avg_f1_for_classes(medium_classes, report)
common_f1 = avg_f1_for_classes(common_classes, report)

print(f"\nRare classes (<100 samples):    F1 = {rare_f1:.4f}")
print(f"Medium classes (100-500):       F1 = {medium_f1:.4f}")
print(f"Common classes (>500 samples):  F1 = {common_f1:.4f}")

print("\n" + "="*50)
print("KEY INSIGHTS:")
print("="*50)
print("✓ PROS of keeping all classes:")
print("  • Model can classify ANY perturbation")
print("  • True generalization - not limited to 25 types")
print("  • No data waste")
print("  • Better reflects real experimental diversity")
print("\n✗ CONS:")
print("  • Lower overall accuracy (but more honest)")
print("  • Rare classes still hard to predict")
print("  • May need more sophisticated approaches")

print("\n💡 RECOMMENDATION:")
print("  Use THIS approach if you need to classify ALL perturbations")
print("  Use FILTERING only if you only care about common ones")
print("  Consider TWO-STAGE model: common vs rare, then classify within each")

ALTERNATIVE: Keep ALL Classes with Class Weighting

🎯 GOAL: Classify ALL perturbations, not just common ones
   Strategy: Use class_weight='balanced' to handle imbalance

📊 Using FULL dataset:
  Total perturbations: 284
  Total samples: 91,205

Reducing 5045 genes -> 200 components...
Explained variance: 48.18%

Training data: (72964, 200)
Test data: (18241, 200)

Training SVM with class_weight='balanced'...
This gives rare classes higher importance during training
[LibLinear]
✓ Training completed in 852.8 seconds

RESULTS: All Classes (No Filtering)
  accuracy:  0.2784
  precision: 0.2517
  recall:    0.2784
  f1_score:  0.2460

COMPARISON: Filtered vs All Classes
Filtered (25 classes):      0.6978
All classes (284 total): 0.2784

Performance by Class Frequency:

Rare classes (<100 samples):    F1 = 0.1017
Medium classes (100-500):       F1 = 0.2492
Common classes (>500 samples):  F1 = 0.3122

KEY INSIGHTS:
✓ PROS of keeping all classes:
  • Model can classify ANY perturbation
  • Tru

In [14]:
# BALANCED APPROACH: Undersample Common Classes
from sklearn.svm import LinearSVC
from sklearn.decomposition import TruncatedSVD
import time

print("="*70)
print("UNDERSAMPLING: Balance Dataset by Limiting Max Samples Per Class")
print("="*70)

# Strategy: Keep all classes, but limit maximum samples per class
max_samples_per_class = 300  # Adjust this threshold

print(f"\n🎯 Strategy: Keep ALL classes, limit each to max {max_samples_per_class} samples")
print("   • Keeps rare classes intact")
print("   • Reduces dominance of common classes")
print("   • More balanced training")

# Create balanced dataset
balanced_indices = []
class_counts_dict = {}

for class_label in np.unique(y):
    # Get indices for this class
    class_indices = np.where(y == class_label)[0]
    class_counts_dict[class_label] = len(class_indices)
    
    # If class has more than max_samples_per_class, randomly sample
    if len(class_indices) > max_samples_per_class:
        np.random.seed(42)
        sampled_indices = np.random.choice(
            class_indices, 
            size=max_samples_per_class, 
            replace=False
        )
        balanced_indices.extend(sampled_indices)
    else:
        # Keep all samples from rare classes
        balanced_indices.extend(class_indices)

balanced_indices = np.array(balanced_indices)

# Create balanced dataset
X_balanced = X[balanced_indices]
y_balanced = y[balanced_indices]

# Show statistics
print("\n" + "="*50)
print("Dataset Statistics:")
print("="*50)
print(f"Original dataset:  {X.shape[0]:,} samples, {len(np.unique(y))} classes")
print(f"Balanced dataset:  {X_balanced.shape[0]:,} samples, {len(np.unique(y_balanced))} classes")
print(f"Reduction: {(1 - X_balanced.shape[0]/X.shape[0])*100:.1f}% fewer samples")

# Show class distribution comparison
balanced_counts = pd.Series(y_balanced).value_counts()
print(f"\nSamples per class (balanced):")
print(f"  Min: {balanced_counts.min()}")
print(f"  Mean: {balanced_counts.mean():.1f}")
print(f"  Max: {balanced_counts.max()}")
print(f"  Std: {balanced_counts.std():.1f}")

# Dimensionality reduction
n_components = 200
print(f"\nReducing {X_balanced.shape[1]} genes -> {n_components} components...")
svd_balanced_under = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced_balanced = svd_balanced_under.fit_transform(X_balanced)
print(f"Explained variance: {svd_balanced_under.explained_variance_ratio_.sum():.2%}")

# Split data
X_train_bal_under, X_test_bal_under, y_train_bal_under, y_test_bal_under = train_test_split(
    X_reduced_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\nTraining data: {X_train_bal_under.shape}")
print(f"Test data: {X_test_bal_under.shape}")

# Train SVM
print("\n" + "="*50)
print("Training SVM on Balanced Data...")
print("="*50)

start_time = time.time()

balanced_under_model = LinearSVC(
    C=1.0,
    loss='squared_hinge',
    dual=False,
    max_iter=2000,
    tol=1e-4,
    random_state=42,
    verbose=1
)

balanced_under_model.fit(X_train_bal_under, y_train_bal_under)

training_time = time.time() - start_time
print(f"\n✓ Training completed in {training_time:.1f} seconds")

# Evaluate
y_pred_bal_under = balanced_under_model.predict(X_test_bal_under)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc_bal_under = accuracy_score(y_test_bal_under, y_pred_bal_under)
prec_bal_under, rec_bal_under, f1_bal_under, _ = precision_recall_fscore_support(
    y_test_bal_under, y_pred_bal_under, average='weighted', zero_division=0
)

print("\n" + "="*50)
print("RESULTS: Undersampled Balanced Data")
print("="*50)
print(f"  accuracy:  {acc_bal_under:.4f}")
print(f"  precision: {prec_bal_under:.4f}")
print(f"  recall:    {rec_bal_under:.4f}")
print(f"  f1_score:  {f1_bal_under:.4f}")

# Comparison with other approaches
print("\n" + "="*50)
print("COMPARISON: Different Balancing Strategies")
print("="*50)
print(f"Filtered (25 classes only):       ", end="")
if 'acc_hp' in locals():
    print(f"{acc_hp:.4f}")
else:
    print("Not run")

print(f"All classes (class_weight):       ", end="")
if 'acc_full' in locals():
    print(f"{acc_full:.4f}")
else:
    print("Not run")

print(f"Undersampled (all {len(np.unique(y_balanced))} classes): {acc_bal_under:.4f}")

print("\n" + "="*50)
print("KEY TRADE-OFFS:")
print("="*50)
print("✓ PROS of undersampling:")
print("  • Keeps ALL classes (true generalization)")
print("  • More balanced training")
print("  • Usually better than class_weight alone")
print("  • Faster training (less data)")
print("\n✗ CONS:")
print("  • Discards data from common classes")
print("  • May lose useful patterns from abundant data")

print("\n💡 TUNING TIP:")
print(f"  Try different max_samples_per_class values:")
print(f"  • {max_samples_per_class}: Current (balanced)")
print(f"  • 500: More data, less balanced")
print(f"  • 200: Very balanced, less data")
print(f"  • 100: Extremely balanced, minimal data")

UNDERSAMPLING: Balance Dataset by Limiting Max Samples Per Class

🎯 Strategy: Keep ALL classes, limit each to max 300 samples
   • Keeps rare classes intact
   • Reduces dominance of common classes
   • More balanced training

Dataset Statistics:
Original dataset:  91,205 samples, 284 classes
Balanced dataset:  69,324 samples, 284 classes
Reduction: 24.0% fewer samples

Samples per class (balanced):
  Min: 49
  Mean: 244.1
  Max: 300
  Std: 68.4

Reducing 5045 genes -> 200 components...
Explained variance: 48.45%

Training data: (55459, 200)
Test data: (13865, 200)

Training SVM on Balanced Data...
[LibLinear]
✓ Training completed in 642.5 seconds

RESULTS: Undersampled Balanced Data
  accuracy:  0.2946
  precision: 0.2464
  recall:    0.2946
  f1_score:  0.2594

COMPARISON: Different Balancing Strategies
Filtered (25 classes only):       0.6978
All classes (class_weight):       0.2784
Undersampled (all 284 classes): 0.2946

KEY TRADE-OFFS:
✓ PROS of undersampling:
  • Keeps ALL classe

Choose one of the following tasks:

**Exploratory Data Analysis and Visualization**

- Objective: Explore the dataset to identify patterns and clusters.
- Tasks:
  - Perform dimensionality reduction using PCA, t-SNE, or UMAP.
  - Visualize gene expression profiles across different conditions or perturbations.
  - Create heatmaps of the top differentially expressed genes.
- Learning Outcomes:
  - Learn to visualize high-dimensional data.
  - Interpret clusters and patterns in the context of biological conditions.

**Machine Learning Classification**

- Objective: Build models to classify samples based on gene expression profiles.
- Tasks:
  - Split the dataset into training and testing sets.
  - Implement classification algorithms.
  - Evaluate model performance using metrics like accuracy, precision, recall, and ROC curves.
- Learning Outcomes:
  - Understand supervised learning techniques.
  - Learn model evaluation and validation strategies.

**Advanced Deep Learning Applications**

- Objective: Apply deep learning techniques to model complex patterns in the data.
- Tasks:
  - Implement autoencoders or variational autoencoders for dimensionality reduction.
  - Explore the use of GANs to generate synthetic gene expression data.
  - Analyze how deep learning models capture nonlinear relationships.
- Learning Outcomes:
  - Gain experience with deep learning frameworks.
  - Understand the applications of deep learning in genomics.

In [5]:
# BEST BALANCED SVM: Good accuracy + reasonable speed
import time

print("="*70)
print("BALANCED SVM: Accuracy vs Speed Tradeoff")
print("="*70)

# Step 1: More aggressive filtering (fewer classes = better accuracy)
min_samples = 500  # Only very common perturbations
unique, counts = np.unique(y_labels, return_counts=True)
common_perturbations = unique[counts >= min_samples]

mask = np.isin(y_labels, common_perturbations)
X_filtered_balanced = X[mask]
y_labels_filtered_balanced = y_labels[mask]

print(f"\nFiltered to {len(common_perturbations)} perturbations (>= {min_samples} samples)")
print(f"Data shape: {X_filtered_balanced.shape}")
print(f"Samples per class: {X_filtered_balanced.shape[0] // len(common_perturbations):.0f} avg")

# Re-encode labels
label_encoder_balanced = LabelEncoder()
y_filtered_balanced = label_encoder_balanced.fit_transform(y_labels_filtered_balanced)

# Step 2: Moderate dimensionality reduction (keep more info)
n_components = 300  # Sweet spot: enough info, not too slow
print(f"\nReducing {X_filtered_balanced.shape[1]} genes -> {n_components} components...")
from sklearn.decomposition import TruncatedSVD
svd_balanced = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced_balanced = svd_balanced.fit_transform(X_filtered_balanced)
print(f"Explained variance: {svd_balanced.explained_variance_ratio_.sum():.2%}")

# Step 3: Split data
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_reduced_balanced, y_filtered_balanced, test_size=0.2, random_state=42, stratify=y_filtered_balanced
)

print(f"Training data: {X_train_bal.shape}")
print(f"Test data: {X_test_bal.shape}")

# Step 4: Train with balanced settings
print("\n" + "="*50)
print("Training Balanced LinearSVC...")
print("="*50)

start_time = time.time()

balanced_model = SVMModel(
    kernel='linear',
    C=1.0,                    # Standard regularization
    pca_components=None,      
    use_sparse=False,         
    random_state=42
)
balanced_model.fit(X_train_bal, y_train_bal)

training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.1f} seconds")

# Step 5: Evaluate
results_balanced = balanced_model.evaluate(X_test_bal, y_test_bal)

print("\n" + "="*50)
print("FINAL Balanced Results:")
print("="*50)
for metric, value in results_balanced.items():
    print(f"  {metric}: {value:.4f}")

print(f"\n⚡ Training time: {training_time:.1f} seconds")
print(f"📊 Number of classes: {len(common_perturbations)}")
print(f"📈 Components used: {n_components}")

# Show what perturbations we're classifying
print(f"\nClassifying these {len(common_perturbations)} perturbations:")
for pert in label_encoder_balanced.inverse_transform(range(len(common_perturbations))):
    print(f"  • {pert}")

BALANCED SVM: Accuracy vs Speed Tradeoff

Filtered to 25 perturbations (>= 500 samples)
Data shape: (22882, 5045)
Samples per class: 915 avg

Reducing 5045 genes -> 300 components...
Explained variance: 60.48%
Training data: (18305, 300)
Test data: (4577, 300)

Training Balanced LinearSVC...
✓ Training completed in 10.7 seconds

FINAL Balanced Results:
  accuracy: 0.6928
  precision: 0.6646
  recall: 0.6928
  f1_score: 0.6552

⚡ Training time: 10.7 seconds
📊 Number of classes: 25
📈 Components used: 300

Classifying these 25 perturbations:
  • BAK1+ctrl
  • CBL+ctrl
  • CEBPE+RUNX1T1
  • DUSP9+ETS2
  • DUSP9+ctrl
  • ETS2+CNN1
  • FOSB+ctrl
  • KLF1+ctrl
  • LHX1+ELMSAN1
  • LYL1+IER5L
  • MAML2+ctrl
  • SET+CEBPE
  • SET+KLF1
  • SET+ctrl
  • SLC4A1+ctrl
  • TBX3+TBX2
  • UBASH3B+OSR2
  • ZC3HAV1+HOXC13
  • ZNF318+ctrl
  • ctrl
  • ctrl+BAK1
  • ctrl+CEBPE
  • ctrl+ETS2
  • ctrl+KLF1
  • ctrl+UBASH3B


In [6]:
# HYPERPARAMETER OPTIMIZATION: Find best SVM settings
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.svm import LinearSVC
from scipy.stats import loguniform, uniform
import time

print("="*70)
print("HYPERPARAMETER OPTIMIZATION (Successive Halving)")
print("="*70)

# Use filtered data with common perturbations
min_samples = 500
unique, counts = np.unique(y_labels, return_counts=True)
common_perturbations = unique[counts >= min_samples]
mask = np.isin(y_labels, common_perturbations)
X_hp = X[mask]
y_labels_hp = y_labels[mask]

label_encoder_hp = LabelEncoder()
y_hp = label_encoder_hp.fit_transform(y_labels_hp)

print(f"Data: {X_hp.shape} | Classes: {len(common_perturbations)}")

# Dimensionality reduction first
n_comp = 200
from sklearn.decomposition import TruncatedSVD
svd_hp = TruncatedSVD(n_components=n_comp, random_state=42)
X_hp_reduced = svd_hp.fit_transform(X_hp)
print(f"Reduced to {n_comp} components")

# Split data
X_train_hp, X_test_hp, y_train_hp, y_test_hp = train_test_split(
    X_hp_reduced, y_hp, test_size=0.2, random_state=42, stratify=y_hp
)

# Define hyperparameter search space
# Note: Only 'squared_hinge' works with dual=False
param_distributions = {
    'C': loguniform(1e-3, 1e2),              # Regularization: 0.001 to 100
    'loss': ['squared_hinge'],               # Only squared_hinge works with dual=False
    'max_iter': [500, 1000, 2000],           # Max iterations
    'tol': loguniform(1e-5, 1e-2),           # Tolerance
}

print("\nHyperparameter search space:")
for param, values in param_distributions.items():
    print(f"  {param}: {values}")

# Create base estimator
base_svm = LinearSVC(
    dual=False,        # dual=False is faster but requires loss='squared_hinge'
    random_state=42,
    verbose=0
)

# Successive Halving Search (like Hyperband)
print("\n" + "="*50)
print("Starting Successive Halving Search...")
print("="*50)

start_time = time.time()

halving_search = HalvingRandomSearchCV(
    base_svm,
    param_distributions,
    n_candidates=30,              # Start with 30 random configs
    factor=3,                     # Keep top 1/3 each iteration
    resource='n_samples',         # Successive halving on sample size
    max_resources='auto',         # Use full dataset at final iteration
    cv=3,                         # 3-fold cross-validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,                    # Use all CPU cores
    verbose=1
)

halving_search.fit(X_train_hp, y_train_hp)

search_time = time.time() - start_time

# Get best parameters
print("\n" + "="*50)
print("BEST HYPERPARAMETERS FOUND:")
print("="*50)
for param, value in halving_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV score: {halving_search.best_score_:.4f}")
print(f"Search time: {search_time:.1f} seconds ({search_time/60:.1f} minutes)")

# Evaluate on test set with best model
y_pred_hp = halving_search.best_estimator_.predict(X_test_hp)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
acc_hp = accuracy_score(y_test_hp, y_pred_hp)
prec_hp, rec_hp, f1_hp, _ = precision_recall_fscore_support(y_test_hp, y_pred_hp, average='weighted')

print("\n" + "="*50)
print("TEST SET PERFORMANCE (Best Model):")
print("="*50)
print(f"  accuracy:  {acc_hp:.4f}")
print(f"  precision: {prec_hp:.4f}")
print(f"  recall:    {rec_hp:.4f}")
print(f"  f1_score:  {f1_hp:.4f}")

# Show iteration details
print("\n" + "="*50)
print("Successive Halving Iterations:")
print("="*50)
results_df = pd.DataFrame(halving_search.cv_results_)
print(results_df[['n_resources', 'mean_test_score', 'rank_test_score']].head(10))

HYPERPARAMETER OPTIMIZATION (Successive Halving)
Data: (22882, 5045) | Classes: 25
Reduced to 200 components

Hyperparameter search space:
  C: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001DDECD54B30>
  loss: ['squared_hinge']
  max_iter: [500, 1000, 2000]
  tol: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001DDECE81D30>

Starting Successive Halving Search...
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 5
min_resources_: 150
max_resources_: 18305
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 30
n_resources: 150
Fitting 3 folds for each of 30 candidates, totalling 90 fits
----------
iter: 1
n_candidates: 10
n_resources: 450
Fitting 3 folds for each of 10 candidates, totalling 30 fits
----------
iter: 2
n_candidates: 4
n_resources: 1350
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 3
n_candidates: 2
n_resources: 4050
Fitting 3 folds for each of 2 candidates

In [7]:
# RBF KERNEL SVM - Hyperparameter Optimization with Successive Halving
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import loguniform, randint
import time

print("="*70)
print("RBF KERNEL SVM - HYPERPARAMETER OPTIMIZATION")
print("="*70)

# Use the same filtered data as linear SVM
print(f"Using filtered data: {X_hp_reduced.shape} | Classes: {len(common_perturbations)}")
print(f"Training set: {X_train_hp.shape}")

# Create pipeline: RBF feature transformation -> Linear SVM
print("\n" + "="*50)
print("Setting up RBF Pipeline...")
print("="*50)

# Build pipeline for RBF kernel approximation
rbf_pipeline = Pipeline([
    ('rbf_features', RBFSampler(random_state=42)),
    ('sgd_svm', SGDClassifier(loss='hinge', penalty='l2', random_state=42, n_jobs=-1))
])

# Define hyperparameter search space for RBF approach
param_distributions_rbf = {
    # RBF feature parameters
    'rbf_features__gamma': loguniform(1e-4, 1e1),      # Kernel width
    'rbf_features__n_components': [300, 500, 700],     # Number of random features
    
    # SGD Classifier parameters
    'sgd_svm__alpha': loguniform(1e-5, 1e-2),          # Regularization (1/C)
    'sgd_svm__max_iter': [1000, 2000, 3000],
    'sgd_svm__tol': loguniform(1e-4, 1e-2),
}

print("Hyperparameter search space:")
for param, values in param_distributions_rbf.items():
    print(f"  {param}: {values}")

# Successive Halving Search
print("\n" + "="*50)
print("Starting Successive Halving Search for RBF...")
print("="*50)

start_time = time.time()

halving_search_rbf = HalvingRandomSearchCV(
    rbf_pipeline,
    param_distributions_rbf,
    n_candidates=30,              # Start with 30 random configs
    factor=3,                     # Keep top 1/3 each iteration
    resource='n_samples',
    max_resources='auto',
    cv=3,                         # 3-fold cross-validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

halving_search_rbf.fit(X_train_hp, y_train_hp)

search_time = time.time() - start_time

# Get best parameters
print("\n" + "="*50)
print("BEST HYPERPARAMETERS FOUND (RBF):")
print("="*50)
for param, value in halving_search_rbf.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV score: {halving_search_rbf.best_score_:.4f}")
print(f"Search time: {search_time:.1f} seconds ({search_time/60:.1f} minutes)")

# Evaluate on test set with best model
y_pred_rbf = halving_search_rbf.best_estimator_.predict(X_test_hp)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc_rbf = accuracy_score(y_test_hp, y_pred_rbf)
prec_rbf, rec_rbf, f1_rbf, _ = precision_recall_fscore_support(
    y_test_hp, y_pred_rbf, average='weighted', zero_division=0
)

print("\n" + "="*50)
print("TEST SET PERFORMANCE (Best RBF Model):")
print("="*50)
print(f"  accuracy:  {acc_rbf:.4f}")
print(f"  precision: {prec_rbf:.4f}")
print(f"  recall:    {rec_rbf:.4f}")
print(f"  f1_score:  {f1_rbf:.4f}")

# Comparison with Linear Kernel
print("\n" + "="*50)
print("FINAL COMPARISON: Linear vs RBF Kernel")
print("="*50)
print(f"Linear SVM accuracy: {acc_hp:.4f}")
print(f"RBF SVM accuracy:    {acc_rbf:.4f}")
print(f"Improvement:         {(acc_rbf - acc_hp):.4f} ({(acc_rbf - acc_hp)*100:.2f}%)")

if acc_rbf > acc_hp + 0.01:  # More than 1% improvement
    print("\n✓ RBF kernel provides significantly better accuracy!")
elif acc_rbf > acc_hp:
    print("\n→ RBF kernel slightly better, but linear is more efficient")
else:
    print("\n→ Linear kernel is sufficient for this data")

# Show iteration details
print("\n" + "="*50)
print("RBF Successive Halving Iterations:")
print("="*50)
results_df_rbf = pd.DataFrame(halving_search_rbf.cv_results_)
print(results_df_rbf[['n_resources', 'mean_test_score', 'rank_test_score']].head(10))

RBF KERNEL SVM - HYPERPARAMETER OPTIMIZATION
Using filtered data: (22882, 200) | Classes: 25
Training set: (18305, 200)

Setting up RBF Pipeline...
Hyperparameter search space:
  rbf_features__gamma: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001DDECE807D0>
  rbf_features__n_components: [300, 500, 700]
  sgd_svm__alpha: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001DDECEB5EB0>
  sgd_svm__max_iter: [1000, 2000, 3000]
  sgd_svm__tol: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001DDED034230>

Starting Successive Halving Search for RBF...
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 5
min_resources_: 150
max_resources_: 18305
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 30
n_resources: 150
Fitting 3 folds for each of 30 candidates, totalling 90 fits
----------
iter: 1
n_candidates: 10
n_resources: 450
Fitting 3 folds for each of 10 candidates, totalling 30 fits

In [12]:
# BAYESIAN OPTIMIZATION - Include min_samples as hyperparameter
# First install: pip install scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder
import time

print("="*70)
print("BAYESIAN OPTIMIZATION with min_samples as hyperparameter")
print("="*70)

# Create custom wrapper that includes filtering
class FilteredSVMClassifier(BaseEstimator, ClassifierMixin):
    """SVM classifier that filters classes by minimum sample count."""
    
    def __init__(self, min_samples=500, C=1.0, loss='squared_hinge', 
                 max_iter=1000, tol=1e-4, random_state=42):
        self.min_samples = min_samples
        self.C = C
        self.loss = loss
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
    
    def fit(self, X, y):
        # Filter classes with min_samples
        unique, counts = np.unique(y, return_counts=True)
        self.valid_classes_ = unique[counts >= self.min_samples]
        
        # Create mask for valid samples
        mask = np.isin(y, self.valid_classes_)
        X_filtered = X[mask]
        y_filtered = y[mask]
        
        # Re-encode labels to consecutive integers
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y_filtered)
        
        # Train LinearSVC
        self.model_ = LinearSVC(
            C=self.C,
            loss=self.loss,
            dual=False,
            max_iter=self.max_iter,
            tol=self.tol,
            random_state=self.random_state
        )
        self.model_.fit(X_filtered, y_encoded)
        
        self.n_classes_kept_ = len(self.valid_classes_)
        self.n_classes_discarded_ = len(unique) - self.n_classes_kept_
        
        return self
    
    def predict(self, X):
        y_pred_encoded = self.model_.predict(X)
        y_pred = self.label_encoder_.inverse_transform(y_pred_encoded)
        return y_pred
    
    def score(self, X, y):
        # Only score on samples that belong to valid classes
        mask = np.isin(y, self.valid_classes_)
        if mask.sum() == 0:
            return 0.0
        
        y_pred = self.predict(X[mask])
        return (y_pred == y[mask]).mean()

print("\n→ Optimizing Linear SVM with min_samples threshold")

# Create the custom classifier
base_model = FilteredSVMClassifier(random_state=42)

# Search space now includes min_samples!
search_spaces = {
    'min_samples': Integer(100, 500),             # Optimize filtering threshold!
    'C': Real(1e-3, 1e2, prior='log-uniform'),
    'loss': Categorical(['squared_hinge']),
    'max_iter': Integer(500, 2000),
    'tol': Real(1e-5, 1e-2, prior='log-uniform'),
}

model_name = "Filtered Linear SVM"

print("\nSearch space (including min_samples!):")
for param, space in search_spaces.items():
    print(f"  {param}: {space}")

# Bayesian Optimization Search
print("\n" + "="*50)
print(f"Starting Bayesian Optimization for {model_name}...")
print("="*50)
print("Note: Using FULL filtered dataset (not pre-reduced)")
print("Bayesian opt will find optimal min_samples threshold!")

start_time = time.time()

# Use the full filtered dataset (before reduction to 200 components)
# We'll use the dataset that was filtered to min_samples=500 earlier
X_full_filtered = X_hp  # Original sparse data with 5045 genes
y_full_filtered = y_hp  # Corresponding labels

# Do dimensionality reduction first (before train/test split)
from sklearn.decomposition import TruncatedSVD
svd_bayes = TruncatedSVD(n_components=200, random_state=42)
X_bayes_reduced = svd_bayes.fit_transform(X_full_filtered)

# Split
X_train_bayes, X_test_bayes, y_train_bayes, y_test_bayes = train_test_split(
    X_bayes_reduced, y_full_filtered, test_size=0.2, random_state=42, stratify=y_full_filtered
)

bayes_search = BayesSearchCV(
    base_model,
    search_spaces,
    n_iter=40,              # Reduced iterations (more params to search)
    cv=3,                   # 3-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,              # Use all CPU cores
    random_state=42,
    verbose=1
)

bayes_search.fit(X_train_bayes, y_train_bayes)

search_time = time.time() - start_time

# Results
print("\n" + "="*50)
print(f"BEST HYPERPARAMETERS FOUND ({model_name}):")
print("="*50)
for param, value in bayes_search.best_params_.items():
    print(f"  {param}: {value}")

# Show how many classes were kept with optimal min_samples
optimal_min_samples = bayes_search.best_params_['min_samples']
print(f"\n📊 With optimal min_samples={optimal_min_samples}:")
print(f"  Classes kept: {bayes_search.best_estimator_.n_classes_kept_}")
print(f"  Classes discarded: {bayes_search.best_estimator_.n_classes_discarded_}")

print(f"\nBest CV score: {bayes_search.best_score_:.4f}")
print(f"Search time: {search_time:.1f} seconds ({search_time/60:.1f} minutes)")

# Test set evaluation
y_pred_bayes = bayes_search.best_estimator_.predict(X_test_bayes)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc_bayes = accuracy_score(y_test_bayes, y_pred_bayes)
prec_bayes, rec_bayes, f1_bayes, _ = precision_recall_fscore_support(
    y_test_bayes, y_pred_bayes, average='weighted', zero_division=0
)

print("\n" + "="*50)
print(f"TEST SET PERFORMANCE (Bayesian {model_name}):")
print("="*50)
print(f"  accuracy:  {acc_bayes:.4f}")
print(f"  precision: {prec_bayes:.4f}")
print(f"  recall:    {rec_bayes:.4f}")
print(f"  f1_score:  {f1_bayes:.4f}")

# Compare with previous methods
print("\n" + "="*50)
print("COMPARISON: Fixed min_samples=500 vs Optimized")
print("="*50)
if 'acc_hp' in locals():
    print(f"Fixed (min_samples=500): {acc_hp:.4f}")
    print(f"Optimized (min_samples={optimal_min_samples}): {acc_bayes:.4f}")
    print(f"Improvement: {(acc_bayes - acc_hp):.4f}")

print("\n💡 Bayesian optimization found the optimal filtering threshold!")
print(f"   Best trade-off: min_samples={optimal_min_samples}")
print(f"   This keeps {bayes_search.best_estimator_.n_classes_kept_} classes with best accuracy")

BAYESIAN OPTIMIZATION with min_samples as hyperparameter

→ Optimizing Linear SVM with min_samples threshold

Search space (including min_samples!):
  min_samples: Integer(low=100, high=500, prior='uniform', transform='identity')
  C: Real(low=0.001, high=100.0, prior='log-uniform', transform='identity')
  loss: Categorical(categories=('squared_hinge',), prior=None)
  max_iter: Integer(low=500, high=2000, prior='uniform', transform='identity')
  tol: Real(low=1e-05, high=0.01, prior='log-uniform', transform='identity')

Starting Bayesian Optimization for Filtered Linear SVM...
Note: Using FULL filtered dataset (not pre-reduced)
Bayesian opt will find optimal min_samples threshold!
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds

In [9]:
# SAVE BEST MODELS AND HYPERPARAMETERS
import pickle
import json
from pathlib import Path

print("="*70)
print("SAVING BEST MODELS & HYPERPARAMETERS")
print("="*70)

# Create models directory if it doesn't exist
models_dir = Path("../models/checkpoints")
models_dir.mkdir(parents=True, exist_ok=True)

# Save results dictionary
results_to_save = {}

# 1. Save Linear SVM (Successive Halving) if it exists
if 'halving_search' in locals():
    print("\n✓ Saving Linear SVM (Successive Halving)...")
    
    # Save the entire search object (includes best estimator)
    with open(models_dir / "linear_svm_halving_search.pkl", "wb") as f:
        pickle.dump(halving_search, f)
    
    # Save best hyperparameters as JSON
    linear_params = {
        "method": "Successive Halving",
        "best_params": halving_search.best_params_,
        "best_cv_score": float(halving_search.best_score_),
        "test_accuracy": float(acc_hp),
        "test_precision": float(prec_hp),
        "test_recall": float(rec_hp),
        "test_f1": float(f1_hp)
    }
    
    with open(models_dir / "linear_svm_halving_params.json", "w") as f:
        json.dump(linear_params, f, indent=2)
    
    results_to_save["linear_svm_halving"] = linear_params
    print(f"  → Saved to {models_dir / 'linear_svm_halving_search.pkl'}")
    print(f"  → Best CV score: {halving_search.best_score_:.4f}")
    print(f"  → Test accuracy: {acc_hp:.4f}")

# 2. Save RBF SVM (Successive Halving) if it exists
if 'halving_search_rbf' in locals():
    print("\n✓ Saving RBF SVM (Successive Halving)...")
    
    with open(models_dir / "rbf_svm_halving_search.pkl", "wb") as f:
        pickle.dump(halving_search_rbf, f)
    
    rbf_params = {
        "method": "Successive Halving",
        "best_params": halving_search_rbf.best_params_,
        "best_cv_score": float(halving_search_rbf.best_score_),
        "test_accuracy": float(acc_rbf),
        "test_precision": float(prec_rbf),
        "test_recall": float(rec_rbf),
        "test_f1": float(f1_rbf)
    }
    
    with open(models_dir / "rbf_svm_halving_params.json", "w") as f:
        json.dump(rbf_params, f, indent=2)
    
    results_to_save["rbf_svm_halving"] = rbf_params
    print(f"  → Saved to {models_dir / 'rbf_svm_halving_search.pkl'}")
    print(f"  → Best CV score: {halving_search_rbf.best_score_:.4f}")
    print(f"  → Test accuracy: {acc_rbf:.4f}")

# 3. Save Bayesian Optimization results if they exist
if 'bayes_search' in locals():
    print("\n✓ Saving Bayesian Optimization results...")
    
    with open(models_dir / "bayesian_search.pkl", "wb") as f:
        pickle.dump(bayes_search, f)
    
    bayes_params = {
        "method": "Bayesian Optimization",
        "model_type": model_name if 'model_name' in locals() else "Unknown",
        "best_params": bayes_search.best_params_,
        "best_cv_score": float(bayes_search.best_score_),
        "test_accuracy": float(acc_bayes),
        "test_precision": float(prec_bayes),
        "test_recall": float(rec_bayes),
        "test_f1": float(f1_bayes)
    }
    
    with open(models_dir / "bayesian_search_params.json", "w") as f:
        json.dump(bayes_params, f, indent=2)
    
    results_to_save["bayesian_opt"] = bayes_params
    print(f"  → Saved to {models_dir / 'bayesian_search.pkl'}")
    print(f"  → Best CV score: {bayes_search.best_score_:.4f}")
    print(f"  → Test accuracy: {acc_bayes:.4f}")

# Save summary of all results
if results_to_save:
    with open(models_dir / "all_results_summary.json", "w") as f:
        json.dump(results_to_save, f, indent=2)
    
    print("\n" + "="*50)
    print("SUMMARY SAVED")
    print("="*50)
    print(f"All results saved to: {models_dir}")
    print("\nTo load later:")
    print("  import pickle")
    print("  with open('models/checkpoints/linear_svm_halving_search.pkl', 'rb') as f:")
    print("      loaded_search = pickle.load(f)")
    print("  best_model = loaded_search.best_estimator_")
else:
    print("\n⚠ No optimization results found to save.")
    print("   Run at least one optimization cell first!")

SAVING BEST MODELS & HYPERPARAMETERS

✓ Saving Linear SVM (Successive Halving)...
  → Saved to ..\models\checkpoints\linear_svm_halving_search.pkl
  → Best CV score: 0.6246
  → Test accuracy: 0.6978

✓ Saving RBF SVM (Successive Halving)...
  → Saved to ..\models\checkpoints\rbf_svm_halving_search.pkl
  → Best CV score: 0.6068
  → Test accuracy: 0.6316

✓ Saving Bayesian Optimization results...
  → Saved to ..\models\checkpoints\bayesian_search.pkl
  → Best CV score: 0.6979
  → Test accuracy: 0.6998

SUMMARY SAVED
All results saved to: ..\models\checkpoints

To load later:
  import pickle
  with open('models/checkpoints/linear_svm_halving_search.pkl', 'rb') as f:
      loaded_search = pickle.load(f)
  best_model = loaded_search.best_estimator_
